In [1]:
"""
MAP - Charting Student Math Misunderstandings - Inference Notebook v2
此版本以 sample_submission.csv 作為格式 ground truth,自動對齊欄位名與 row_id。
"""
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==== 路徑 ====
BASE_MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"
ADAPTER_PATH = "/kaggle/input/datasets/alextsai2004/gemma-math-misunderstanding-lora/best_gemma_lora_model"
TEST_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
TRAIN_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
SAMPLE_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/sample_submission.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

BATCH_SIZE =16
NUM_BEAMS = 5
NUM_RETURN = 3
MAX_NEW_TOKENS = 32


# ==== STEP 0: 先檢查 sample_submission 的格式,這是 ground truth ====
print("=" * 60)
print("Sample submission inspection:")
print("=" * 60)
sample = pd.read_csv(SAMPLE_CSV)
print(f"Columns: {sample.columns.tolist()}")
print(f"Dtypes:\n{sample.dtypes}")
print(f"Row count: {len(sample)}")
print(f"First 5 rows:\n{sample.head()}")
print(f"\nFirst value of pred col: {repr(sample.iloc[0, 1])}")
print("=" * 60)

# 動態抓出 row_id 跟 prediction 欄位名(不要 hardcode)
ROW_ID_COL = sample.columns[0]
PRED_COL = sample.columns[1]
print(f"Using ROW_ID_COL='{ROW_ID_COL}', PRED_COL='{PRED_COL}'")


# ==== Prompt(必須跟訓練時完全一致) ====
def build_user_prompt(question, correct_answer, student_explanation):
    return (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "If the category is not a misconception type, use `NA` for the misconception part.\n\n"
        f"Question: {question}\n"
        f"Correct answer: {correct_answer}\n"
        f"Student explanation: {student_explanation}\n\n"
        "Return only the label."
    )


def build_prompt(row):
    user = build_user_prompt(
        row["QuestionText"], row["MC_Answer"], row["StudentExplanation"]
    )
    return f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n"


# ==== 載入 model ====
print("Loading base model in bf16 ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Attaching LoRA adapter ...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
stop_ids = [tokenizer.eos_token_id]
if end_of_turn_id and end_of_turn_id != tokenizer.unk_token_id:
    stop_ids.append(end_of_turn_id)


# ==== Fallback labels(訓練集最常見的三個) ====
train_df = pd.read_csv(TRAIN_CSV)
train_df["target"] = (
    train_df["Category"].astype(str) + ":" +
    train_df["Misconception"].fillna("NA").astype(str)
)
fallback_labels = train_df["target"].value_counts().head(3).index.tolist()
print(f"Fallback labels: {fallback_labels}")


# ==== Test ====
test_df = pd.read_csv(TEST_CSV)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    test_df[col] = test_df[col].fillna("")
print(f"Test size: {len(test_df)}")
assert len(test_df) == len(sample), \
    f"test.csv 列數 {len(test_df)} != sample_submission 列數 {len(sample)}"


# ==== 批次推論 ====
def clean_label(text):
    """避免 label 含空白破壞 space-delimited 格式"""
    if not text:
        return ""
    label = text.splitlines()[0].strip()
    if " " in label:
        label = label.split(" ")[0]
    return label


def decode_one_beam(beam_ids, prompt_len):
    gen = beam_ids[prompt_len:]
    text = tokenizer.decode(gen, skip_special_tokens=True).strip()
    return clean_label(text)


# 用 dict 累積 row_id -> 預測字串,最後再 map 到 sample 順序
pred_dict = {}

for start in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Inference"):
    batch = test_df.iloc[start:start + BATCH_SIZE]
    prompts = [build_prompt(r) for _, r in batch.iterrows()]

    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            num_return_sequences=NUM_RETURN,
            do_sample=False,
            early_stopping=True,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    prompt_len = enc["input_ids"].shape[1]
    outputs = outputs.view(len(batch), NUM_RETURN, -1)

    for i, (_, row) in enumerate(batch.iterrows()):
        labels = []
        for k in range(NUM_RETURN):
            label = decode_one_beam(outputs[i, k], prompt_len)
            if label and label not in labels:
                labels.append(label)
        for fb in fallback_labels:
            if len(labels) >= 3:
                break
            if fb not in labels:
                labels.append(fb)
        while len(labels) < 3:
            labels.append(labels[0] if labels else "True_Correct:NA")
        pred_dict[row["row_id"]] = " ".join(labels[:3])


# ==== 用 sample 作為骨架建 submission,保證欄位名跟 row_id 順序對 ====
submission = sample.copy()
submission[PRED_COL] = submission[ROW_ID_COL].map(pred_dict)

# ==== 驗證 ====
print("\n" + "=" * 60)
print("Submission validation:")
print("=" * 60)
print(f"Shape: {submission.shape}")
print(f"Columns: {submission.columns.tolist()}")
print(f"Any NaN: {submission.isna().any().to_dict()}")
print(f"Any empty string: {(submission[PRED_COL] == '').sum()}")
print(f"Pred col dtype: {submission[PRED_COL].dtype}")
print(f"Sample predictions:\n{submission.head()}")

assert submission.shape == sample.shape, "shape 跟 sample 不一致"
assert not submission.isna().any().any(), "有 NaN!"
assert (submission[PRED_COL] != "").all(), "有空字串!"
assert (submission[PRED_COL].str.split().str.len() == 3).all(), \
    "不是每筆都恰好 3 個 token"

submission.to_csv(OUTPUT_CSV, index=False)
print(f"\n[OK] Saved {OUTPUT_CSV}")

Sample submission inspection:
Columns: ['row_id', 'Category:Misconception']
Dtypes:
row_id                     int64
Category:Misconception    object
dtype: object
Row count: 3
First 5 rows:
   row_id                             Category:Misconception
0   36696  True_Correct:NA False_Neither:NA False_Misconc...
1   36697  True_Correct:NA False_Neither:NA False_Misconc...
2   36698  True_Correct:NA False_Neither:NA False_Misconc...

First value of pred col: 'True_Correct:NA False_Neither:NA False_Misconception:Incomplete'
Using ROW_ID_COL='row_id', PRED_COL='Category:Misconception'
Loading base model in bf16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Attaching LoRA adapter ...


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Fallback labels: ['True_Correct:NA', 'False_Neither:NA', 'True_Neither:NA']
Test size: 3


Inference: 100%|██████████| 1/1 [00:03<00:00,  3.80s/it]


Submission validation:
Shape: (3, 2)
Columns: ['row_id', 'Category:Misconception']
Any NaN: {'row_id': False, 'Category:Misconception': False}
Any empty string: 0
Pred col dtype: object
Sample predictions:
   row_id                             Category:Misconception
0   36696  True_Neither:NA True_Correct:NA True_Misconcep...
1   36697  False_Neither:NA False_Misconception:WNB False...
2   36698  True_Neither:NA True_Correct:NA True_Misconcep...

[OK] Saved /kaggle/working/submission.csv
